In [1355]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

e = 1e-15

In [1356]:
np.random.seed(42)

In [1357]:
data = pd.read_csv("data.csv")
dataset = data.to_numpy()
dataset

array([[842302, 'M', 17.99, ..., 0.4601, 0.1189, nan],
       [842517, 'M', 20.57, ..., 0.275, 0.08902, nan],
       [84300903, 'M', 19.69, ..., 0.3613, 0.08758, nan],
       ...,
       [926954, 'M', 16.6, ..., 0.2218, 0.0782, nan],
       [927241, 'M', 20.6, ..., 0.4087, 0.124, nan],
       [92751, 'B', 7.76, ..., 0.2871, 0.07039, nan]], dtype=object)

In [1358]:
dataset[dataset[:,1] == 'M'][:,1] = 1

In [1359]:
dataset[dataset[:,1] == 'B'][:,1] = 0

In [1360]:
dataset[:,1] = (dataset[:,1] == 'M').astype(int)

In [1361]:
dataset

array([[842302, 1, 17.99, ..., 0.4601, 0.1189, nan],
       [842517, 1, 20.57, ..., 0.275, 0.08902, nan],
       [84300903, 1, 19.69, ..., 0.3613, 0.08758, nan],
       ...,
       [926954, 1, 16.6, ..., 0.2218, 0.0782, nan],
       [927241, 1, 20.6, ..., 0.4087, 0.124, nan],
       [92751, 0, 7.76, ..., 0.2871, 0.07039, nan]], dtype=object)

In [1362]:
data.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [1363]:
dataset.shape

(569, 33)

In [1364]:
dataset = dataset[:,:32]

In [1365]:
training_set = dataset[:455]
testing_set = dataset[455:]
training_set.shape,testing_set.shape

((455, 32), (114, 32))

In [1366]:
ouput_training = training_set[:,1].reshape(455,1).T
input_training = training_set[:,2:].T
ouput_training.shape,input_training.shape

((1, 455), (30, 455))

In [1367]:
input_training = (input_training/input_training.mean( axis=1, keepdims=True)).astype(float).round(7)

In [1368]:
input_training

array([[1.2637556, 1.4449946, 1.3831767, ..., 0.8429721, 1.0206987,
        0.8865256],
       [0.546618 , 0.9357806, 1.1190398, ..., 1.4866115, 0.7361965,
        0.9031309],
       [1.3246515, 1.4336009, 1.4023184, ..., 0.8281229, 1.0124739,
        0.8696531],
       ...,
       [2.2642359, 1.586842 , 2.0731323, ..., 0.607095 , 0.9120076,
        0.8404291],
       [1.5645145, 0.9351043, 1.2285571, ..., 0.8320728, 0.8861389,
        1.111924 ],
       [1.4152214, 1.0595712, 1.0424314, ..., 0.9753006, 0.9295946,
        0.872462 ]])

In [1369]:
def init_params(input_size,hidden_size,output_size):
    w1 = np.random.normal(size=(hidden_size,input_size)).round(5)
    b1 = np.random.normal(size=(hidden_size,1)).round(5)
    w2 = np.random.normal(size=(output_size,hidden_size)).round(5)
    b2 = np.random.normal(size=(output_size,1)).round(5)
    
    return w1,b1,w2,b2

In [1370]:
def relu(x : int):
    return np.maximum(x,0)

def sigmoid(x : float):
    return (1 + np.exp(-x))**-1

In [1371]:
def forward_prop(inputf :np.array , w1:np.array,b1:np.array,w2:np.array,b2:np.array):
    w1_dash = np.dot(w1,inputf)
    z1 = w1_dash + np.broadcast_to(b1,shape=w1_dash.shape)
    a1 = relu(z1)
    w2_dash = np.dot(w2,a1)
    z2 = w2_dash + np.broadcast_to(b2,shape=w2_dash.shape)
    a2 = sigmoid(z2)
    return( z1 , a1 , z2 , a2 )

In [1372]:
def compute_cost(a2 : np.array , y : np.array):
    a2 = np.clip(a2,e,1-e)
    cost = -np.divide(1,y.shape[1])*np.sum( y*np.log(a2) + (1-y)*np.log(1-a2))
    return cost

In [1373]:
def back_propogation( inputf:np.array , outputf:np.array , z1:np.array , a1:np.array , z2:np.array , a2:np.array , w1:np.array , w2:np.array ):
    m = inputf.shape[1]
    dz2 = a2 - outputf
    dw2 = (1/m)*np.dot(dz2,a1.T)
    db2 = (1/m)*np.sum(dz2, axis=1,keepdims=True)
    dz1 = np.dot(w2.T,dz2)*((z1 > 0).astype(int))
    dw1 = (1/m)*np.dot(dz1,inputf.T)
    db1 = (1/m)*np.sum(dz1, axis=1, keepdims=True)
    return ( dw1 , db1 , dw2 , db2 )

In [1374]:
def update_params(w1: np.array, b1: np.array, w2: np.array, b2: np.array, dw1: np.array, db1: np.array, dw2: np.array, db2: np.array, eta: float = 0.01):
    w1 -= (eta * dw1).astype(float)
    b1 -= (eta * db1).astype(float)
    w2 -= (eta * dw2).astype(float)
    b2 -= (eta * db2).astype(float)
    
    return w1, b1, w2, b2

In [1375]:
def get_accuracy(predictions: np.array, y_true: np.array) -> float:
    binary_predictions = (predictions > 0.5).astype(int)
    correct_predictions = (binary_predictions == y_true)
    accuracy = np.mean(correct_predictions) * 100
    
    return np.squeeze(accuracy)

In [1376]:
w1,b1,w2,b2 = init_params(30,16,1)
epochs = int(input("Enter the number of epoches you want to train over : "))
for i in tqdm(range(epochs+1)):
    z1 , a1 , z2 , a2 = forward_prop(input_training,w1,b1,w2,b2)
    if( i%100 == 0):
        accuracy = get_accuracy(a2, ouput_training)
        cost = compute_cost(a2,ouput_training)
        print(f"Epoch {i} | Cost: {cost:.4f} | Accuracy: {accuracy:.2f}%")
    dw1, db1, dw2, db2 = back_propogation(input_training,ouput_training,z1,a1,z2,a2,w1,w2)
    w1, b1, w2, b2 = update_params(w1,b1,w2,b2,dw1,db1,dw2,db2,eta=0.01)

  3%|▎         | 27/1001 [00:00<00:07, 130.15it/s]

Epoch 0 | Cost: 2.0284 | Accuracy: 50.99%


 12%|█▏        | 125/1001 [00:00<00:06, 133.88it/s]

Epoch 100 | Cost: 0.4529 | Accuracy: 82.86%


 22%|██▏       | 223/1001 [00:01<00:06, 128.55it/s]

...

Epoch 800 | Cost: 0.1717 | Accuracy: 93.41%


 92%|█████████▏| 917/1001 [00:07<00:00, 127.46it/s]

Epoch 900 | Cost: 0.1629 | Accuracy: 93.41%


100%|██████████| 1001/1001 [00:07<00:00, 127.63it/s]

Epoch 1000 | Cost: 0.1551 | Accuracy: 93.19%


In [1377]:
testing_set

array([[9112085, 0, 13.38, ..., 0.07763, 0.2196, 0.07675],
       [9112366, 0, 11.63, ..., 0.06835, 0.2884, 0.0722],
       [9112367, 0, 13.21, ..., 0.06005, 0.2444, 0.06788],
       ...,
       [926954, 1, 16.6, ..., 0.1418, 0.2218, 0.0782],
       [927241, 1, 20.6, ..., 0.265, 0.4087, 0.124],
       [92751, 0, 7.76, ..., 0.0, 0.2871, 0.07039]], dtype=object)

In [1378]:
testing_set = testing_set[:,1:]

In [1379]:
testing_set

array([[0, 13.38, 30.72, ..., 0.07763, 0.2196, 0.07675],
       [0, 11.63, 29.29, ..., 0.06835, 0.2884, 0.0722],
       [0, 13.21, 25.25, ..., 0.06005, 0.2444, 0.06788],
       ...,
       [1, 16.6, 28.08, ..., 0.1418, 0.2218, 0.0782],
       [1, 20.6, 29.33, ..., 0.265, 0.4087, 0.124],
       [0, 7.76, 24.54, ..., 0.0, 0.2871, 0.07039]], dtype=object)

In [1380]:
input_testing  = testing_set[:,1:].T
input_testing

array([[13.38, 11.63, 13.21, ..., 16.6, 20.6, 7.76],
       [30.72, 29.29, 25.25, ..., 28.08, 29.33, 24.54],
       [86.34, 74.87, 84.1, ..., 108.3, 140.1, 47.92],
       ...,
       [0.07763, 0.06835, 0.06005, ..., 0.1418, 0.265, 0.0],
       [0.2196, 0.2884, 0.2444, ..., 0.2218, 0.4087, 0.2871],
       [0.07675, 0.0722, 0.06788, ..., 0.0782, 0.124, 0.07039]],
      dtype=object)

In [1381]:
ouput_testing = testing_set[:,0]
ouput_testing.shape

(114,)

In [1382]:
ouput_testing = ouput_testing.reshape(1,114)

In [1383]:
input_testing.shape,ouput_testing.shape

((30, 114), (1, 114))

In [1384]:
input_testing = input_testing.astype(np.float64)

In [1385]:
input_testing = (input_testing/np.mean(input_testing,axis=1).reshape(30,1))

In [1386]:
z1 , a1 , z2 , a2 = forward_prop(input_testing,w1,b1,w2,b2)

In [1387]:
acc =get_accuracy(a2,ouput_testing)
print(f"Testing accuracy : {acc}")

Testing accuracy : 90.35087719298247


In [1388]:
a2

array([[1.86316427e-01, 2.25624452e-02, 1.89732307e-02, 4.30082869e-02,
        4.74851693e-03, 9.99800150e-01, 1.00000000e+00, 3.61583517e-02,
        1.10200481e-02, 3.47437929e-01, 3.81251290e-01, 9.05694603e-01,
        1.26660964e-03, 9.99998851e-01, 7.25752741e-01, 9.38964728e-03,
        1.59686561e-01, 4.44404039e-01, 1.48789098e-04, 6.77464186e-02,
        1.04170449e-01, 8.24770343e-01, 7.86411917e-03, 7.37211894e-03,
        9.88353988e-01, 3.02970835e-02, 1.31004638e-01, 1.71754027e-01,
        1.90440523e-01, 8.89870567e-01, 3.74686880e-02, 1.40282654e-01,
        9.99960754e-01, 1.82053088e-01, 1.14137994e-02, 9.38754344e-03,
        3.76852328e-01, 9.98967654e-01, 5.86071489e-04, 4.79064186e-03,
        2.34485135e-01, 7.53416796e-01, 1.07072370e-01, 9.99980510e-01,
        9.99996781e-01, 8.67409138e-01, 9.10770722e-01, 2.10465571e-02,
        9.99999981e-01, 2.18272875e-01, 9.09193159e-03, 5.69628522e-02,
        2.71184876e-03, 4.56818533e-01, 9.98058063e-01, 1.910487